# Hermes voice agent — Qwen3-ASR → Llama (Ollama) → gestalt world

Speak a command; a local Llama model acts on a sandbox 3D world. Everything runs on the Colab GPU — no API keys, no cloud LLM.

```
  mic ──► Qwen3-ASR (GPU) ──► text ──► Llama 3.1 8B via Ollama (GPU)
                                              │  tool calls (MCP, stdio)
                                              ▼
                                     gestalt-mcp server
                                     (sandbox world: add / move /
                                      recolor / describe objects)
                                              │
                                              ▼  /content/world.json ──► 3D view
```

**Why Colab:** both Qwen3-ASR and the Llama model run locally on the GPU. Set the runtime to **GPU** (Runtime → Change runtime type → GPU) first. On a free T4 (16 GB), Qwen3-ASR-0.6B (~1.5 GB) + Llama 3.1 8B Q4 (~5 GB) fit together.

The agent turn is `run_turn(...)` (in `gestalt.ollama_agent`) — it doesn't care whether the command was typed or transcribed, which is the seam a *Hermes* agent will wrap. The gestalt MCP server is LLM-agnostic; only this notebook's agent layer talks to Ollama.

## 1. Runtime & install

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached
%pip install -q -U "qwen-asr" "ollama>=0.4" "mcp>=1.20" soundfile

In [ ]:
# Install the gestalt package (provides `gestalt.mcp_server` and the Ollama bridge).
import os
if not os.path.exists('gestalt'):
    !git clone -q https://github.com/davechendatascience/gestalt.git
%pip install -q -e ./gestalt

## 2. Install Ollama and pull a Llama model

Installs the Ollama runtime, starts the server in the background, and pulls a tool-capable Llama. `llama3.1:8b` has solid tool calling and fits the T4; switch `OLLAMA_MODEL` to `llama3.2:3b` for a lighter/faster option.

In [ ]:
import os, subprocess, time, urllib.request

OLLAMA_MODEL = 'llama3.1:8b'  # or 'llama3.2:3b'

# 1) install the Ollama runtime
!curl -fsSL https://ollama.com/install.sh | sh

# 2) serve in the background (persists for the kernel session)
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['ollama', 'serve'])

# 3) wait for the server to accept connections
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print('Ollama is up.'); break
    except Exception:
        time.sleep(1)

# 4) pull the model (first run downloads a few GB)
!ollama pull {OLLAMA_MODEL}

## 3. Load Qwen3-ASR on the GPU

`Qwen/Qwen3-ASR-0.6B` is small and fast; switch `ASR_MODEL_ID` to `Qwen/Qwen3-ASR-1.7B` for higher accuracy. First run downloads the weights.

In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

ASR_MODEL_ID = 'Qwen/Qwen3-ASR-0.6B'  # or 'Qwen/Qwen3-ASR-1.7B'
asr = Qwen3ASRModel.from_pretrained(
    ASR_MODEL_ID,
    dtype=torch.bfloat16,   # float16 also works (e.g. on T4)
    device_map='cuda:0',
    max_new_tokens=256,
)

def transcribe(audio):
    """audio: a file path, URL, or (np.ndarray, sample_rate) tuple."""
    results = asr.transcribe(audio=audio, language=None)  # language=None -> auto-detect
    return results[0].text.strip()

print('ASR ready:', ASR_MODEL_ID)

## 4. Capture a voice command

`record(seconds)` records from your browser mic; `upload()` takes an audio file instead. Both return a 16 kHz mono WAV path. (You can skip this section and just type commands in step 5.)

In [ ]:
import subprocess
from base64 import b64decode
from IPython.display import Javascript, display

_RECORD_JS = '''
async function recordAudio(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(res => rec.onstop = res);
  rec.start();
  await new Promise(r => setTimeout(r, ms));
  rec.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  const buf = await new Blob(chunks).arrayBuffer();
  const bytes = new Uint8Array(buf);
  let binary = '';
  for (let i = 0; i < bytes.length; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}
'''

def _to_wav(src, out_path='/content/command.wav'):
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src,
                    '-ar', '16000', '-ac', '1', out_path], check=True)
    return out_path

def record(seconds=5, out_path='/content/command.wav'):
    from google.colab import output
    display(Javascript(_RECORD_JS))
    print(f'Recording {seconds}s — speak now...')
    b64 = output.eval_js(f'recordAudio({int(seconds * 1000)})')
    with open('/content/command.webm', 'wb') as f:
        f.write(b64decode(b64))
    print('Transcribing...')
    return _to_wav('/content/command.webm', out_path)

def upload(out_path='/content/command.wav'):
    from google.colab import files
    up = files.upload()
    return _to_wav(next(iter(up)), out_path)

In [ ]:
# Try it: speak something like
#   "add a red cube called box at one two zero, then a blue sphere above it"
wav = record(6)
command = transcribe(wav)
print('Heard:', command)

## 5. Wire the local Llama to the gestalt world (MCP over stdio)

`run_command(text)` launches the `gestalt-mcp` server as a subprocess, exposes its tools to the Llama model, and runs one turn (the model calls tools, we execute them via MCP, feed results back). The world is persisted to `/content/world.json`, so state survives across calls and we can visualize it. A short conversation history is kept for references like *"move it up"*.

In [ ]:
import os
from ollama import AsyncClient
from gestalt.ollama_agent import run_turn

os.environ['GESTALT_WORLD_FILE'] = '/content/world.json'
if os.path.exists('/content/world.json'):
    os.remove('/content/world.json')  # start empty

ollama_client = AsyncClient()  # talks to the server started in step 2
history = []

async def run_command(text):
    return await run_turn(
        ollama_client, OLLAMA_MODEL, history, text,
        on_tool=lambda name, args: print(f'  · {name}({args})'),
    )

In [ ]:
# Deterministic demo (no mic needed) — verifies the whole Llama -> MCP loop.
reply = await run_command('Add a red cube named box at (1, 2, 0) and a blue sphere named ball at (1, 2, 3). How far apart are they?')
print('Hermes:', reply)

In [ ]:
# Run the command you spoke in step 4:
print('Hermes:', await run_command(command))

## 6. Visualize the world

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def show_world(path='/content/world.json'):
    try:
        objs = json.load(open(path))['objects']
    except FileNotFoundError:
        print('No world yet — run a command first.'); return
    if not objs:
        print('The world is empty.'); return
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    for o in objs:
        c = o.get('color') or 'gray'
        if not mcolors.is_color_like(c):
            c = 'gray'
        ax.scatter(o['x'], o['y'], o['z'], s=220, c=c, edgecolors='k', depthshade=True)
        ax.text(o['x'], o['y'], o['z'], f"  {o['name']} ({o['kind']})")
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_title(f'gestalt world — {len(objs)} object(s)')
    plt.show()

show_world()

## 7. Full voice loop

Run this cell, speak a command, and watch the world update. Re-run it for each new command.

In [ ]:
async def voice_turn(seconds=6):
    text = transcribe(record(seconds))
    print('Heard:', text)
    print('Hermes:', await run_command(text))
    show_world()

await voice_turn()

## Where Hermes plugs in

- **Integration seam:** `run_turn(...)` (here wrapped as `run_command`) is the agent turn. A Hermes agent wraps this — feeding it transcribed (or typed) commands and surfacing the reply — without changing anything below it.
- **LLM-agnostic tools:** `gestalt-mcp` is a standard MCP server; swapping Claude for a local Llama only touched `gestalt.ollama_agent`, not the server. It runs over **stdio** here but can be exposed over HTTP/SSE and shared by any MCP client.
- **Ground truth lives in the world**, not the chat: `/content/world.json` is the state; `describe_scene` lets the model re-ground at any time.

Run the same loop with typed input on your own machine (Ollama serving locally): `pip install -e ".[agent]"` then `python examples/text_agent.py`.